In [21]:
!pip install -q langgraph langchain google-generativeai pydantic faiss-cpu sentence-transformers python-dotenv langsmith langchain-google-genai


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import os
import json
import datetime
from typing import Dict, List, Optional, Any
from enum import Enum
from pydantic import BaseModel, Field


from dotenv import load_dotenv
load_dotenv()  


os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "true")
os.environ["LANGCHAIN_ENDPOINT"] = os.getenv("LANGCHAIN_ENDPOINT", "https://api.smith.langchain.com")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "AI-detector")

import google.generativeai as genai
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from langgraph.graph import StateGraph, END

if os.environ["LANGCHAIN_API_KEY"]:
    print(f"LangSmith tracing ENABLED for project: {os.environ['LANGCHAIN_PROJECT']}")
else:
    print("LangSmith API key missing. Tracing disabled. Add LANGCHAIN_API_KEY to .env")

API_KEY = os.getenv("GOOGLE_API_KEY")
if not API_KEY:
    API_KEY = input("Please enter your Google Gemini API key: ")

genai.configure(api_key=API_KEY)

try:
    test_model = genai.GenerativeModel("models/gemini-2.5-flash")
    test_response = test_model.generate_content("OK")
    print("Gemini API key works!")
except Exception as e:
    print(f"⚠️ API key error: {e}")
    print("Please check your .env file or the key you entered.")

class SuspectStatus(str, Enum):
    ALIVE = "alive"
    DEAD = "dead"
    MISSING = "missing"

class SuspectState(BaseModel):
    name: str
    status: SuspectStatus = SuspectStatus.ALIVE
    statements: List[str] = Field(default_factory=list)
    alibi: Optional[str] = None
    hidden_secret: str = ""
    relationship_to_victim: str = ""
    last_interaction: Optional[str] = None

class TimelineEvent(BaseModel):
    timestamp: str
    description: str
    location: str
    source: str  

class InvestigationState(BaseModel):
    discovered_evidence: List[str] = Field(default_factory=list)
    suspect_states: Dict[str, SuspectState] = Field(default_factory=dict)
    timeline: List[TimelineEvent] = Field(default_factory=list)
    current_location: str = "Mansion Entrance"
    user_inventory: List[str] = Field(default_factory=list)
    action_history: List[str] = Field(default_factory=list)
    
    
    contradictions: List[str] = Field(default_factory=list)
    last_agent_output: str = ""
    graph_step_count: int = 0
    user_input: str = ""  

print("Imports, LangSmith, and state ready.")

LangSmith tracing ENABLED for project: AI-detector

Gemini API key works!

Imports, LangSmith, and state ready.

In [23]:
from langsmith.wrappers import wrap_gemini
from langsmith import traceable

wrapped_genai = wrap_gemini(genai)

class GeminiLLM:
    def __init__(self, model_name="models/gemini-2.5-flash"):
        self.model = wrapped_genai.GenerativeModel(model_name)
        self.model_name = model_name
    
    @traceable(name="GeminiLLM.generate", run_type="llm")
    def generate(self, prompt: str, temperature=0.7) -> str:
        try:
            response = self.model.generate_content(prompt, generation_config={"temperature": temperature})
            from langsmith import get_current_run_tree
            if run := get_current_run_tree():
                run.add_metadata({
                    "model": self.model_name,
                    "temperature": temperature,
                    "prompt_length": len(prompt)
                })
            return response.text.strip()
        except Exception as e:
            print(f"[LLM Error] {e}")
            return "I'm sorry, I couldn't process that request."

llm = GeminiLLM()
print("LLM with LangSmith tracing ready.")

LLM with LangSmith tracing ready.

In [24]:
class FaissEvidenceStore:
    def __init__(self):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.index = None
        self.evidence_texts = []
    
    def build_index(self, evidence_list):
        self.evidence_texts = evidence_list
        if not evidence_list:
            return
        embeddings = self.model.encode(evidence_list)
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(embeddings.astype(np.float32))
    
    def search(self, query, k=3):
        if self.index is None or not self.evidence_texts:
            return []
        query_vec = self.model.encode([query]).astype(np.float32)
        distances, indices = self.index.search(query_vec, min(k, len(self.evidence_texts)))
        results = [self.evidence_texts[i] for i in indices[0] if i != -1]
        return results

evidence_corpus = [
    "Broken pocket watch found near the body, stopped at 10:15 PM.",
    "Muddy footprints leading from the garden to the library.",
    "A letter threatening Lord Ashworth, signed 'A Friend'.",
    "Butler's handkerchief with blood stains found in his quarters.",
    "Lady Evelyn's diary entry: 'He will pay for what he did.'",
    "A half‑empty bottle of poison in the study drawer.",
    "Phone records showing a call at 10:05 PM from the victim's study to an unknown number."
]

evidence_store = FaissEvidenceStore()
evidence_store.build_index(evidence_corpus)
print("Faiss evidence store ready.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10107.47it/s]


Faiss evidence store ready.

In [25]:
class EvidenceLookupTool:
    def __init__(self, store: FaissEvidenceStore):
        self.store = store
    
    @traceable(name="EvidenceLookupTool.run", run_type="tool")
    def run(self, query: str) -> List[str]:
        """Search for evidence relevant to the query."""
        return self.store.search(query)

class TimelineQueryTool:
    @traceable(name="TimelineQueryTool.run", run_type="tool")
    def run(self, state: InvestigationState, time_query: str) -> str:
        """Return timeline events matching a time (e.g., '9 PM', 'evening')."""
        results = []
        for event in state.timeline:
            if time_query.lower() in event.timestamp.lower() or time_query.lower() in event.description.lower():
                results.append(f"{event.timestamp}: {event.description} ({event.location})")
        if not results:
            return f"No events found matching '{time_query}'."
        return "\n".join(results)

evidence_tool = EvidenceLookupTool(evidence_store)
timeline_tool = TimelineQueryTool()
print("Tools ready.")

Tools ready.

In [26]:
class NarratorOutput(BaseModel):
    description: str
    scene_update: str

class NarratorAgent:
    def __init__(self, llm: GeminiLLM):
        self.llm = llm
    
    @traceable(name="NarratorAgent.run", run_type="chain")
    def run(self, state: InvestigationState, user_input: str) -> NarratorOutput:
        prompt = f"""
You are the narrator of a murder mystery: "The Midnight Murder at Lord Ashworth's Mansion".
Current location: {state.current_location}
Discovered evidence: {state.discovered_evidence}
Recent action history: {state.action_history[-3:]}
User action: {user_input}

Provide a short immersive description (2-3 sentences) of what happens next and update the scene if needed.
Output as JSON with keys: "description", "scene_update" (new location or progress note).
"""
        result = self.llm.generate(prompt)
        try:
            clean_result = result.strip()
            if clean_result.startswith("```json"):
                clean_result = clean_result[7:]  
            if clean_result.startswith("```"):
                clean_result = clean_result[3:]  
            if clean_result.endswith("```"):
                clean_result = clean_result[:-3]  
            data = json.loads(clean_result.strip())
            return NarratorOutput(**data)
        except Exception as e:
            print(f"[Narrator parse error] {e}\nRaw: {result}")
            return NarratorOutput(description=result, scene_update="No major change.")

narrator = NarratorAgent(llm)
print("Narrator Agent ready.")

Narrator Agent ready.

In [27]:
class SuspectResponse(BaseModel):
    dialogue: str
    hidden_hint: Optional[str] = None
    updated_alibi: Optional[str] = None

class SuspectAgent:
    def __init__(self, llm: GeminiLLM, suspect_name: str):
        self.llm = llm
        self.suspect_name = suspect_name
    
    @traceable(name="SuspectAgent.run", run_type="chain")
    def run(self, state: InvestigationState, user_question: str) -> SuspectResponse:
        suspect_state = state.suspect_states.get(self.suspect_name)
        if not suspect_state:
            raise ValueError(f"Unknown suspect: {self.suspect_name}")
        if suspect_state.status != SuspectStatus.ALIVE:
            return SuspectResponse(
                dialogue=f"{self.suspect_name} is {suspect_state.status.value} and cannot be interviewed.",
                hidden_hint=None
            )
        
        prompt = f"""
You are {self.suspect_name}, a suspect in the murder of Lord Ashworth.
Your personality: {suspect_state.relationship_to_victim}
Your secret: {suspect_state.hidden_secret}
Your previous statements: {suspect_state.statements[-3:]}
Known evidence the detective has: {state.discovered_evidence}
Detective asks: "{user_question}"

Respond in character, staying consistent with your secret. You may reveal a slight hint if pressured.
Output JSON: {{"dialogue": "...", "hidden_hint": "optional hint", "updated_alibi": "new info if any"}}
"""
        result = self.llm.generate(prompt)
        try:
            data = json.loads(result)
            return SuspectResponse(**data)
        except:
            return SuspectResponse(dialogue=result, hidden_hint=None)

suspect_butler = SuspectAgent(llm, "Mr. Barnes (Butler)")
suspect_lady = SuspectAgent(llm, "Lady Evelyn")
print("Suspect Agents ready.")

Suspect Agents ready.

In [28]:
class EvidenceOutput(BaseModel):
    found_evidence: List[str]
    new_discoveries: List[str]
    summary: str

class EvidenceManager:
    def __init__(self, lookup_tool: EvidenceLookupTool):
        self.lookup_tool = lookup_tool
    
    @traceable(name="EvidenceManager.run", run_type="chain")
    def run(self, state: InvestigationState, user_query: str) -> EvidenceOutput:
        relevant = self.lookup_tool.run(user_query)
        new_items = [item for item in relevant if item not in state.discovered_evidence]
        return EvidenceOutput(
            found_evidence=relevant,
            new_discoveries=new_items,
            summary=f"Found {len(relevant)} relevant pieces of evidence. New: {len(new_items)}."
        )

evidence_manager = EvidenceManager(evidence_tool)
print("Evidence Manager ready.")

Evidence Manager ready.

In [29]:
class ValidationResult(BaseModel):
    is_consistent: bool
    contradictions: List[str]
    suggestion: str

class ConsistencyValidator:
    def __init__(self, llm: GeminiLLM):
        self.llm = llm
    
    @traceable(name="ConsistencyValidator.run", run_type="chain")
    def run(self, state: InvestigationState) -> ValidationResult:
        suspect_summary = "\n".join([f"{name}: {st.statements[-1] if st.statements else 'No statement'}" 
                                     for name, st in state.suspect_states.items()])
        timeline_summary = "\n".join([f"{ev.timestamp}: {ev.description}" for ev in state.timeline])
        
        prompt = f"""
You are a Consistency Validator for a detective game.
Check for contradictions in:
Suspect statements:
{suspect_summary}
Timeline events:
{timeline_summary}
Discovered evidence:
{state.discovered_evidence}

Output JSON: 
{{"is_consistent": true/false, "contradictions": ["list of contradictions"], "suggestion": "advice for detective"}}
"""
        result = self.llm.generate(prompt)
        try:
            data = json.loads(result)
            return ValidationResult(**data)
        except:
            return ValidationResult(is_consistent=True, contradictions=[], suggestion="No clear contradictions.")

validator = ConsistencyValidator(llm)
print("Consistency Validator ready.")

Consistency Validator ready.

In [30]:
def classify_intent(user_input: str):
    lower = user_input.lower()
    if "interview" in lower or "ask" in lower:
        for name in ["butler", "lady evelyn", "evelyn", "barnes"]:
            if name in lower:
                return "suspect"
    if "evidence" in lower or "clue" in lower or "search" in lower:
        return "evidence"
    if "contradiction" in lower or "inconsistent" in lower:
        return "consistency"
    if "timeline" in lower or "where was" in lower or "what time" in lower:
        return "timeline_tool"
    return "narrator"

def build_graph():
    workflow = StateGraph(InvestigationState)
    
    def narrator_node(state: InvestigationState):
        out = narrator.run(state, state.user_input)
        new_state = state.copy(deep=True)
        new_state.last_agent_output = out.description
        new_state.action_history.append(f"Narrator: {out.description}")
        new_state.graph_step_count += 1
        if out.scene_update and out.scene_update != "No major change.":
            new_state.current_location = out.scene_update
        return new_state
    
    def suspect_node(state: InvestigationState):
        if "butler" in state.user_input.lower() or "barnes" in state.user_input.lower():
            agent = suspect_butler
            name = "Mr. Barnes (Butler)"
        else:
            agent = suspect_lady
            name = "Lady Evelyn"
        out = agent.run(state, state.user_input)
        new_state = state.copy(deep=True)
        new_state.suspect_states[name].statements.append(out.dialogue)
        if out.updated_alibi:
            new_state.suspect_states[name].alibi = out.updated_alibi
        new_state.last_agent_output = out.dialogue
        new_state.action_history.append(f"Interviewed {name}: {out.dialogue[:80]}...")
        new_state.graph_step_count += 1
        return new_state
    
    def evidence_node(state: InvestigationState):
        out = evidence_manager.run(state, state.user_input)
        new_state = state.copy(deep=True)
        for item in out.new_discoveries:
            if item not in new_state.discovered_evidence:
                new_state.discovered_evidence.append(item)
        new_state.last_agent_output = out.summary
        new_state.action_history.append(f"Evidence lookup: {out.summary}")
        new_state.graph_step_count += 1
        return new_state
    
    def timeline_node(state: InvestigationState):
        result = timeline_tool.run(state, state.user_input)
        new_state = state.copy(deep=True)
        new_state.last_agent_output = result
        new_state.action_history.append(f"Timeline query: {state.user_input[:50]}")
        new_state.graph_step_count += 1
        return new_state
    
    def consistency_node(state: InvestigationState):
        out = validator.run(state)
        new_state = state.copy(deep=True)
        if not out.is_consistent:
            new_state.contradictions = out.contradictions
        new_state.last_agent_output = f"Consistency check: {out.suggestion}"
        new_state.action_history.append(f"Validator: {out.suggestion}")
        new_state.graph_step_count += 1
        return new_state
    
    workflow.add_node("narrator", narrator_node)
    workflow.add_node("suspect", suspect_node)
    workflow.add_node("evidence", evidence_node)
    workflow.add_node("timeline_tool", timeline_node)
    workflow.add_node("consistency", consistency_node)
    
    def route(state: InvestigationState):
        return classify_intent(state.user_input)
    
    workflow.set_conditional_entry_point(route, {
        "narrator": "narrator",
        "suspect": "suspect",
        "evidence": "evidence",
        "timeline_tool": "timeline_tool",
        "consistency": "consistency"
    })
    
    workflow.add_edge("suspect", "consistency")
    workflow.add_edge("evidence", "consistency")
    workflow.add_edge("consistency", END)
    workflow.add_edge("narrator", END)
    workflow.add_edge("timeline_tool", END)
    
    return workflow.compile()

graph_app = build_graph()
print("LangGraph compiled.")

LangGraph compiled.

In [31]:
initial_state = InvestigationState(
    discovered_evidence=["Broken pocket watch found near the body, stopped at 10:15 PM."],
    suspect_states={
        "Mr. Barnes (Butler)": SuspectState(
            name="Mr. Barnes (Butler)",
            status=SuspectStatus.ALIVE,
            alibi="I was in the kitchen from 9 PM to 10 PM.",
            hidden_secret="I owed Lord Ashworth a large gambling debt. He threatened to fire me.",
            relationship_to_victim="Employer-employee, resentful."
        ),
        "Lady Evelyn": SuspectState(
            name="Lady Evelyn",
            status=SuspectStatus.ALIVE,
            alibi="I was reading in the library from 9:30 PM to 10:30 PM.",
            hidden_secret="Lord Ashworth had an affair and threatened divorce. I would lose everything.",
            relationship_to_victim="Wife, unhappy marriage."
        )
    },
    timeline=[
        TimelineEvent(timestamp="9:00 PM", description="Lord Ashworth seen alive in study", location="Study", source="narrator"),
        TimelineEvent(timestamp="10:15 PM", description="Gunshot heard by servants", location="Mansion", source="narrator"),
        TimelineEvent(timestamp="10:20 PM", description="Body discovered", location="Study", source="narrator")
    ],
    current_location="Mansion Entrance",
    user_inventory=[],
    action_history=[],
    contradictions=[],
    graph_step_count=0
)

# Simple logging helper
def log_transition(state: InvestigationState):
    print(f"[LOG] Step {state.graph_step_count} | Location: {state.current_location} | Evidence: {len(state.discovered_evidence)} | Contradictions: {len(state.contradictions)}")
    print(f"      Last output: {state.last_agent_output[:100]}...\n")

print("World initialized.")

World initialized.

In [32]:
print("\n" + "="*60)
print("🕵️  The Midnight Murder at Lord Ashworth's Mansion")
print("="*60)
print("You are a detective. Interview suspects, search for evidence, check contradictions, query timelines.")
print("Type 'quit' to exit.\n")

current_state = initial_state.copy(deep=True)

while True:
    user_cmd = input("🔍 Detective> ").strip()
    if user_cmd.lower() in ["quit", "exit"]:
        print("Case closed. Well done, detective.")
        break
    
    current_state.user_input = user_cmd
    
    try:
        new_state = graph_app.invoke(current_state.dict())
       
        current_state = InvestigationState(**new_state)
        print(f"\n📢 {current_state.last_agent_output}\n")
        log_transition(current_state)
    except Exception as e:
        print(f"Error: {e}. Please rephrase your action.\n")
        
        current_state.action_history.append(f"FAILED: {user_cmd}")

============================================================

🕵️  The Midnight Murder at Lord Ashworth's Mansion

============================================================

You are a detective. Interview suspects, search for evidence, check contradictions, query timelines.

Type 'quit' to exit.

...

[Narrator parse error] Expecting value: line 1 column 1 (char 0)
Raw: I'm sorry, I couldn't process that request.

📢 I'm sorry, I couldn't process that request.

[LOG] Step 6 | Location: Foyer: Examining Lord Ashworth's other hand; discovered a folded parchment. | Evidence: 1 
| Contradictions: 0

Last output: I'm sorry, I couldn't process that request....

Case closed. Well done, detective.